# ShellWhisperer-1.5B: Shell Command Prediction

**Model:** Qwen2.5-Coder-1.5B-Instruct | **VRAM:** ~6GB | **Time:** ~2h

In [ ]:
# C1: Install + GPU
import subprocess, sys, torch
from pathlib import Path
subprocess.check_call([sys.executable,'-m','pip','install','--no-cache-dir','unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git'])
subprocess.check_call([sys.executable,'-m','pip','install','--no-deps','trl>=0.12.0'])
extras=['transformers>=4.46.0','datasets>=3.0.0','accelerate>=1.1.0','peft>=0.13.0','bitsandbytes>=0.44.0','torch>=2.1.0','huggingface_hub>=0.24.0','tqdm','sentencepiece','onnxruntime','onnx']
subprocess.check_call([sys.executable,'-m','pip','install','--no-cache-dir',*extras])
import unsloth; print(f'Unsloth {unsloth.__version__}')
import trl; print(f'TRL {trl.__version__}')
if not torch.cuda.is_available(): raise RuntimeError('No GPU!')
print(f'GPU: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB')
OUTPUT_DIR=Path('/content/shell-whisperer-1.5b'); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
HF_USERNAME='your-username'; HF_REPO_ID=f'{HF_USERNAME}/ShellWhisperer-1.5B'


In [ ]:
# C2: Load model
from unsloth import FastLanguageModel
import gc
torch.cuda.empty_cache();gc.collect()
model,tokenizer=FastLanguageModel.from_pretrained(model_name='unsloth/Qwen2.5-Coder-1.5B-Instruct',max_seq_length=2048,load_in_4bit=True,dtype=None,trust_remote_code=True)
model=FastLanguageModel.get_peft_model(model,r=16,lora_alpha=32,lora_dropout=0,bias='none',use_gradient_checkpointing='unsloth',random_state=42,use_rslora=False,loftq_config=None,target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
print(f'Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} / {sum(p.numel() for p in model.parameters()):,}')
print(f'VRAM: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB')


In [ ]:
# C3: Data
from datasets import Dataset
from tqdm.notebook import tqdm
import random
random.seed(42)
CMDS=[('find all python files larger than 1mb','find . -name "*.py" -size +1M'),('show last 5 git commits','git log --oneline -5'),('stop all docker containers','docker stop $(docker ps -q)'),('check process on port 8080','lsof -i :8080'),('count lines in python files','find . -name "*.py" -exec cat {} + | wc -l'),('delete all pyc files','find . -name "*.pyc" -delete'),('list recently modified files','find . -mtime -1 -type f'),('show disk usage','du -sh ~/* | sort -rh | head -10'),('kill highest memory process','ps aux --sort=-%mem | head -2 | tail -1 | awk "{print $2}" | xargs kill'),('download with resume','curl -C - -O https://example.com/file.zip')]
samples=[]
for desc,cmd in CMDS:
    for _ in range(300):
        prefix=random.choice(['','In the project directory, ','I need to '])
        samples.append({'conversations':[{'role':'system','content':'You are ShellWhisperer. Output only the command.'},{'role':'user','content':f'{prefix}{desc}'},{'role':'assistant','content':cmd}]})
random.shuffle(samples)
shell_ds=Dataset.from_list(samples)
print(f'Dataset: {len(shell_ds):,} samples')
def fmt(ex):
    return {'text':tokenizer.apply_chat_template(ex['conversations'],tokenize=False,add_generation_prompt=False)}
shell_formatted=shell_ds.map(fmt)
print(f'Formatted: {len(shell_formatted):,} samples')


In [ ]:
# C4: SFT Training
from trl import SFTTrainer
from transformers import TrainingArguments
torch.cuda.empty_cache();gc.collect()
trainer=SFTTrainer(model=model,tokenizer=tokenizer,train_dataset=shell_formatted,dataset_text_field='text',max_seq_length=2048,args=TrainingArguments(output_dir=str(OUTPUT_DIR/'checkpoints'),learning_rate=2e-4,num_train_epochs=1,per_device_train_batch_size=4,gradient_accumulation_steps=4,warmup_steps=25,logging_steps=10,save_steps=100,save_total_limit=3,optim='adamw_8bit',weight_decay=0.01,lr_scheduler_type='cosine',seed=42,fp16=not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_bf16_supported(),report_to='tensorboard'))
try: r=trainer.train(); print(f'Done! Loss: {r.training_loss:.4f}')
except RuntimeError as e:
    if 'out of memory' in str(e).lower(): torch.cuda.empty_cache();gc.collect();trainer.args.per_device_train_batch_size=2;trainer.args.gradient_accumulation_steps=8;r=trainer.train();print(f'Done (recovery)! Loss: {r.training_loss:.4f}')
    else: raise
model.save_pretrained(str(OUTPUT_DIR/'checkpoints'));tokenizer.save_pretrained(str(OUTPUT_DIR/'checkpoints'))
print(f'VRAM: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB')


In [ ]:
# C5: Evaluate
FastLanguageModel.for_inference(model)
for p in ['find all python files larger than 1mb','show last 5 git commits','stop all docker containers','check process on port 8080','count lines in python files']:
    msgs=[{'role':'system','content':'You are ShellWhisperer. Output only the command.'},{'role':'user','content':p}]
    inp=tokenizer.apply_chat_template(msgs,tokenize=True,add_generation_prompt=True,return_tensors='pt').to('cuda')
    with torch.no_grad(): out=model.generate(input_ids=inp,max_new_tokens=128,temperature=0.3,top_p=0.9,do_sample=True)
    resp=tokenizer.decode(out[0][inp.shape[1]:],skip_special_tokens=True).strip()
    print(f'Q: {p}\nA: {resp}\n')


In [ ]:
# C6: Export GGUF
from pathlib import Path
EXPORT_DIR=OUTPUT_DIR/'exports'; EXPORT_DIR.mkdir(parents=True,exist_ok=True)
merged_dir=str(EXPORT_DIR/'shell-whisperer-1.5b-16bit')
model.save_pretrained_merged(merged_dir,tokenizer,save_method='merged_16bit')
gguf_dir=str(EXPORT_DIR/'shell-whisperer-1.5b-gguf')
try: model.save_pretrained_gguf(gguf_dir,tokenizer,quantization_method='q4_k_m')
except: model.save_pretrained_gguf(gguf_dir,tokenizer,quantization_method='q8_0')
lora_dir=str(EXPORT_DIR/'shell-whisperer-1.5b-lora')
model.save_pretrained(lora_dir);tokenizer.save_pretrained(lora_dir)
print(f'Exports: 16-bit, GGUF, LoRA saved')


In [ ]:
# C7: ONNX export
import onnxruntime as ort
FastLanguageModel.for_inference(model)
onnx_path=str(EXPORT_DIR/'shell_whisperer_1.5b.onnx')
model_cpu=model.cpu()
dummy=tokenizer('List all Python files',return_tensors='pt')
try:
    torch.onnx.export(model_cpu,(dummy['input_ids'],),onnx_path,input_names=['input_ids'],output_names=['logits'],dynamic_axes={'input_ids':{0:'batch',1:'seq_len'},'logits':{0:'batch',1:'seq_len'}},opset_version=14)
    print(f'ONNX exported: {onnx_path}')
    sess=ort.InferenceSession(onnx_path)
    print(f'ONNX valid: inputs={[i.name for i in sess.get_inputs()]}')
except Exception as e: print(f'ONNX failed: {e}. Use GGUF instead.')
model.to('cuda')


In [ ]:
# C9: Upload
from huggingface_hub import HfApi,create_repo
import os
try: from google.colab import userdata; HF_TOKEN=userdata.get('HF_TOKEN')
except: HF_TOKEN=os.environ.get('HF_TOKEN',None) or input('HF token: ')
if HF_TOKEN:
    api=HfApi(token=HF_TOKEN); create_repo(repo_id=HF_REPO_ID,exist_ok=True,token=HF_TOKEN)
    api.upload_folder(folder_path=merged_dir,repo_id=HF_REPO_ID,token=HF_TOKEN,commit_message='ShellWhisperer-1.5B 16-bit')
    lr=f'{HF_REPO_ID}-LoRA'; create_repo(repo_id=lr,exist_ok=True,token=HF_TOKEN)
    api.upload_folder(folder_path=lora_dir,repo_id=lr,token=HF_TOKEN,commit_message='ShellWhisperer-1.5B LoRA')
    if os.path.exists(gguf_dir):
        gr=f'{HF_REPO_ID}-GGUF'; create_repo(repo_id=gr,exist_ok=True,token=HF_TOKEN)
        for gf in Path(gguf_dir).glob('*.gguf'): api.upload_file(path_or_fileobj=str(gf),path_in_repo=gf.name,repo_id=gr,token=HF_TOKEN)
    print(f'https://huggingface.co/{HF_REPO_ID}')
else: print('No HF token')
print('ShellWhisperer-1.5B Training Complete!')
